# Flight Analytics SQL Queries

Each query below answers one of the requested flight analytics problems.

In [ ]:
import mysql.connector
import pandas as pd

from Flight_analytics_database import get_database_config


def run_query(query):
    connection = mysql.connector.connect(**get_database_config(prompt=True))
    cursor = connection.cursor(dictionary=True)
    try:
        cursor.execute(query)
        columns = [description[0] for description in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)
    finally:
        cursor.close()
        connection.close()


print('SQL query helper ready')

SQL query helper ready


## 1. Total flights for each aircraft model

In [2]:
query_01 = """
SELECT COALESCE(a.model, f.aircraft_model, 'Unknown') AS aircraft_model,
       COUNT(*) AS flight_count
FROM flights AS f
LEFT JOIN aircraft AS a
       ON a.registration = f.aircraft_registration
GROUP BY COALESCE(a.model, f.aircraft_model, 'Unknown')
ORDER BY flight_count DESC, aircraft_model
"""
run_query(query_01)

,aircraft_model,flight_count
0,Airbus A320 NEO,3197
1,Airbus A320,3145
2,Airbus A321 NEO,1933
3,Boeing 737-800,1233
4,Boeing 777-300ER,1086
...,...,...
100,Cessna 525A Citation CJ2,1
101,Cessna 680 Citation Latitudde,1
102,Douglas DC-9-30 Freighter,1
103,Embraer Pheom 300,1


## 2. Aircraft assigned to more than 5 flights

In [3]:
query_02 = """
SELECT f.aircraft_registration AS registration,
       COALESCE(a.model, f.aircraft_model, 'Unknown') AS aircraft_model,
       COUNT(*) AS flight_count
FROM flights AS f
LEFT JOIN aircraft AS a
       ON a.registration = f.aircraft_registration
WHERE NULLIF(TRIM(f.aircraft_registration), '') IS NOT NULL
GROUP BY f.aircraft_registration, COALESCE(a.model, f.aircraft_model, 'Unknown')
HAVING COUNT(*) > 5
ORDER BY flight_count DESC, registration
"""
run_query(query_02)

,registration,aircraft_model,flight_count
0,VT-AYC,Boeing 737-800,17
1,VT-GSI,Embraer 175,17
2,VT-NHT,Airbus A321 NEO,17
3,VT-ATH,Airbus A320 NEO,16
4,VT-BWL,Boeing 737 MAX 8,16
...,...,...,...
550,VT-TNW,Airbus A320 NEO,6
551,VT-TNY,Airbus A320 NEO,6
552,VT-TQW,Airbus A320 NEO,6
553,VT-TVA,Airbus A321 NEO,6


## 3. Airports with more than 5 outbound flights

In [4]:
query_03 = """
SELECT COALESCE(a.name, f.origin_iata, 'Unknown') AS airport_name,
       f.origin_iata AS iata_code,
       COUNT(*) AS outbound_flights
FROM flights AS f
LEFT JOIN airport AS a ON a.iata_code = f.origin_iata
GROUP BY f.origin_iata, a.name
HAVING COUNT(*) > 5
ORDER BY outbound_flights DESC, airport_name
"""
run_query(query_03)

,airport_name,iata_code,outbound_flights
0,New Delhi Indira Gandhi,DEL,1574
1,London Heathrow,LHR,1453
2,Paris Charles de Gaulle,CDG,1428
3,New York John F Kennedy,JFK,1124
4,Dubai,DXB,999
...,...,...,...
333,PUJ,PUJ,6
334,SAV,SAV,6
335,TAO,TAO,6
336,TRN,TRN,6


## 4. Top 3 destination airports by arriving flights

In [5]:
query_04 = """
SELECT COALESCE(a.name, f.destination_iata, 'Unknown') AS airport_name,
       a.city,
       f.destination_iata AS iata_code,
       COUNT(*) AS arriving_flights
FROM flights AS f
LEFT JOIN airport AS a ON a.iata_code = f.destination_iata
WHERE f.destination_iata IS NOT NULL
GROUP BY f.destination_iata, a.name, a.city
ORDER BY arriving_flights DESC, airport_name
LIMIT 3
"""
run_query(query_04)

,airport_name,city,iata_code,arriving_flights
0,New Delhi Indira Gandhi,New Delhi,DEL,295
1,Mumbai Chhatrapati Shivaji,Mumbai,BOM,279
2,Bangalore Bengaluru,Bangalore,BLR,226


## 5. Domestic or international flight classification

In [6]:
query_05 = """
SELECT f.flight_number,
       f.origin_iata AS origin,
       f.destination_iata AS destination,
       CASE
           WHEN origin_airport.country IS NULL OR destination_airport.country IS NULL
           THEN 'Unknown'
           WHEN origin_airport.country = destination_airport.country
           THEN 'Domestic'
           ELSE 'International'
       END AS flight_type
FROM flights AS f
LEFT JOIN airport AS origin_airport
     ON origin_airport.iata_code = f.origin_iata
LEFT JOIN airport AS destination_airport
     ON destination_airport.iata_code = f.destination_iata
ORDER BY f.flight_number
"""
run_query(query_05)

,flight_number,origin,destination,flight_type
0,1I 240,JFK,TEB,Unknown
1,1I 954,JFK,TEB,Unknown
2,2K 7392,GYE,NaN,Unknown
3,2K 7392,GYE,NaN,Unknown
4,2K 7393,JFK,GYE,Unknown
...,...,...,...,...
21482,ZZ 845,DEL,NaN,Unknown
21483,ZZ 900,DEL,NaN,Unknown
21484,ZZ 900,DEL,NaN,Unknown
21485,ZZ 930,DEL,NaN,Unknown


## 6. Five most recent arrivals at DEL

In [7]:
query_06 = """
SELECT f.flight_number,
       COALESCE(NULLIF(f.aircraft_registration, ''), f.aircraft_model, 'Unknown') AS aircraft,
       COALESCE(origin_airport.name, f.origin_iata, 'Unknown') AS departure_airport,
       f.actual_arrival AS arrival_time
FROM flights AS f
LEFT JOIN airport AS origin_airport
     ON origin_airport.iata_code = f.origin_iata
WHERE f.destination_iata = 'DEL'
  AND f.actual_arrival IS NOT NULL
ORDER BY arrival_time DESC
LIMIT 5
"""
run_query(query_06)

,flight_number,aircraft,departure_airport,arrival_time
0,VS 300,Boeing 787-9,London Heathrow,2026-09-26 09:10:00
1,BA 257,Boeing 787-8,London Heathrow,2026-09-26 08:00:00
2,BA 137,Boeing 787-9,London Heathrow,2026-09-26 04:59:00
3,AI 2284,Airbus A320 NEO,Doha Hamad,2026-09-26 04:45:00
4,FZ 441,Boeing 737-800,Dubai,2026-09-26 03:10:00


## 7. Airports with no arriving flights

An empty result means every airport currently stored in the `airport` table has at least one arriving flight.

In [8]:
query_07 = """
SELECT a.name AS airport_name,
       a.city,
       a.iata_code
FROM airport AS a
WHERE NOT EXISTS (
    SELECT 1
    FROM flights AS f
    WHERE f.destination_iata = a.iata_code
)
ORDER BY airport_name
"""
run_query(query_07)

,airport_name,city,iata_code


## 8. Flight status counts for each airline

In [9]:
query_08 = """
SELECT COALESCE(NULLIF(airline_code, ''), 'Unknown') AS airline_code,
       COUNT(*) AS total_flights,
       SUM(CASE WHEN status = 'On Time' THEN 1 ELSE 0 END) AS on_time_flights,
       SUM(CASE WHEN status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_flights,
       SUM(CASE WHEN status IN ('Cancelled', 'Canceled') THEN 1 ELSE 0 END) AS cancelled_flights,
       SUM(CASE WHEN status = 'Arrived' THEN 1 ELSE 0 END) AS arrived_flights,
       SUM(CASE WHEN status = 'Departed' THEN 1 ELSE 0 END) AS departed_flights,
       SUM(CASE WHEN status = 'Expected' THEN 1 ELSE 0 END) AS expected_flights,
       SUM(CASE WHEN status NOT IN ('On Time', 'Delayed', 'Cancelled', 'Canceled', 'Arrived', 'Departed', 'Expected') THEN 1 ELSE 0 END) AS other_status_flights
FROM flights
GROUP BY COALESCE(NULLIF(airline_code, ''), 'Unknown')
ORDER BY airline_code
"""
run_query(query_08)

,airline_code,total_flights,on_time_flights,delayed_flights,cancelled_flights,arrived_flights,departed_flights,expected_flights,other_status_flights
0,1I,2,0,0,0,0,1,1,0
1,2K,10,0,0,0,4,3,3,0
2,2Y,2,0,0,0,1,1,0,0
3,3F,4,0,0,0,2,2,0,0
4,3L,13,0,0,0,1,5,5,2
...,...,...,...,...,...,...,...,...,...
258,Z0,3,0,0,0,1,1,1,0
259,ZA,4,0,0,4,0,0,0,0
260,ZG,7,0,0,0,2,2,3,0
261,ZH,26,0,0,0,7,9,7,3


## 9. Cancelled flights with aircraft and airports

In [10]:
query_09 = """
SELECT f.flight_number,
       COALESCE(NULLIF(f.aircraft_registration, ''), f.aircraft_model, 'Unknown') AS aircraft,
       COALESCE(origin_airport.name, f.origin_iata, 'Unknown') AS departure_airport,
       COALESCE(destination_airport.name, f.destination_iata, 'Unknown') AS arrival_airport,
       f.scheduled_departure AS departure_time
FROM flights AS f
LEFT JOIN airport AS origin_airport
     ON origin_airport.iata_code = f.origin_iata
LEFT JOIN airport AS destination_airport
     ON destination_airport.iata_code = f.destination_iata
WHERE f.status IN ('Cancelled', 'Canceled')
ORDER BY departure_time DESC
"""
run_query(query_09)

,flight_number,aircraft,departure_airport,arrival_airport,departure_time
0,8B 156,Airbus A321,Singapore Changi,CGK,2026-09-26 23:20:00
1,EK 390,Boeing 777-300ER,Dubai,HKT,2026-09-26 22:40:00
2,TG 225,Airbus A320,Bangkok Suvarnabhumi,HKT,2026-09-26 22:40:00
3,TR 621,Airbus A321 NEO,Bangkok Suvarnabhumi,Singapore Changi,2026-09-26 22:30:00
4,IB 48,Airbus A320,London Heathrow,MAD,2026-09-26 22:30:00
...,...,...,...,...,...
476,KY 8367,Boeing 737-800,KMG,Unknown,NaT
477,VZ 3733,Boeing 737,MFM,Unknown,NaT
478,QR 413,Unknown,DAM,Unknown,NaT
479,FD 4401,Airbus A320,KBV,Unknown,NaT


## 10. City pairs with more than 2 aircraft models

In [11]:
query_10 = """
SELECT origin_airport.city AS origin_city,
       destination_airport.city AS destination_city,
       COUNT(DISTINCT COALESCE(a.model, f.aircraft_model)) AS aircraft_model_count
FROM flights AS f
LEFT JOIN airport AS origin_airport
     ON origin_airport.iata_code = f.origin_iata
LEFT JOIN airport AS destination_airport
     ON destination_airport.iata_code = f.destination_iata
LEFT JOIN aircraft AS a
     ON a.registration = f.aircraft_registration
WHERE origin_airport.city IS NOT NULL AND destination_airport.city IS NOT NULL
GROUP BY origin_airport.city, destination_airport.city
HAVING COUNT(DISTINCT COALESCE(a.model, f.aircraft_model)) > 2
ORDER BY aircraft_model_count DESC, origin_city, destination_city
"""
run_query(query_10)

,origin_city,destination_city,aircraft_model_count
0,New York,London,13
1,New Delhi,Mumbai,12
2,London,New York,11
3,Mumbai,New Delhi,11
4,Dubai,Mumbai,9
...,...,...,...
99,Paris,New Delhi,3
100,Paris,Singapore,3
101,Singapore,Bangalore,3
102,Singapore,Hyderabad,3


## 11. Delayed-flight percentage by destination airport

In [12]:
query_11 = """
SELECT COALESCE(a.name, f.destination_iata, 'Unknown') AS airport_name,
       f.destination_iata AS iata_code,
       COUNT(*) AS total_arrivals,
       SUM(CASE WHEN f.status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_arrivals,
       ROUND(100.0 * SUM(CASE WHEN f.status = 'Delayed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS delayed_percentage
FROM flights AS f
LEFT JOIN airport AS a ON a.iata_code = f.destination_iata
WHERE f.destination_iata IS NOT NULL
GROUP BY f.destination_iata, a.name
ORDER BY delayed_percentage DESC, airport_name
"""
run_query(query_11)

,airport_name,iata_code,total_arrivals,delayed_arrivals,delayed_percentage
0,BEN,BEN,2,1,50.00
1,MSQ,MSQ,3,1,33.33
2,VVO,VVO,3,1,33.33
3,ASB,ASB,4,1,25.00
4,LJU,LJU,6,1,16.67
...,...,...,...,...,...
671,ZNZ,ZNZ,2,0,0.00
672,ZQN,ZQN,7,0,0.00
673,ZRH,ZRH,55,0,0.00
674,ZTH,ZTH,2,0,0.00
